# Predictive Modeling — ATD Forecasting

This notebook builds and evaluates a predictive model for **Actual Time of Delivery (ATD)**.

## Approach
- **Target**: `ATD_capped` (minutes, Winsorized at p99 to reduce outlier influence)
- **Train/Test Split**: time-based (last 2 weeks as holdout) to simulate production deployment
- **Baseline Models**: global median, territory × time_block segment median
- **Primary Model**: LightGBM (gradient boosting — handles categoricals natively, fast on 1M rows)
- **Evaluation**: MAE, RMSE, MAPE, R² — both globally and by segment

---
**Sections**
1. Setup & Data Load
2. Target Variable Analysis
3. Time-Based Train / Test Split
4. Feature Matrix Construction
5. Baseline Models
6. LightGBM Model
7. Evaluation & Benchmarking
8. Feature Importance
9. Residual & Error Analysis
10. Model Persistence & Summary

---
## 1. Setup & Data Load

In [ ]:
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})
SEED = 42

RAW_PATH      = Path('../data/raw/BC_A&A_with_ATD.csv')
FEATURES_PATH = Path('../data/processed/features.parquet')
MODEL_DIR     = Path('../model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — excludes zeros in y_true."""
    mask = y_true > 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate(y_true, y_pred, label='Model'):
    """Print standard regression metrics."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mpe  = mape(np.array(y_true), np.array(y_pred))
    print(f'{label:30s}  MAE={mae:.2f} min  RMSE={rmse:.2f} min  MAPE={mpe:.1f}%  R²={r2:.4f}')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mpe, 'R2': r2}

In [ ]:
if FEATURES_PATH.exists():
    df = pd.read_parquet(FEATURES_PATH)
    print(f'Loaded processed features: {df.shape}')
else:
    # Build features from scratch
    dtype_map = {
        'region': 'category', 'territory': 'category',
        'country_name': 'category', 'courier_flow': 'category',
        'geo_archetype': 'category', 'merchant_surface': 'category',
    }
    df = pd.read_csv(
        RAW_PATH, dtype=dtype_map,
        parse_dates=['restaurant_offered_timestamp_utc',
                     'order_final_state_timestamp_local',
                     'eater_request_timestamp_local'],
        na_values=['\\N', 'NULL', 'None', ''],
    )
    df = df[df['ATD'] > 0].copy()
    df['ATD_capped'] = df['ATD'].clip(upper=df['ATD'].quantile(0.99))

    ts = df['restaurant_offered_timestamp_utc']
    df['hour_utc']     = ts.dt.hour
    df['hour_local']   = (df['hour_utc'] - 6) % 24
    df['day_of_week']  = ts.dt.dayofweek
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_peak_hour'] = df['hour_local'].isin(list(range(12,15)) + list(range(19,24))).astype(int)
    df['week_number']  = ts.dt.isocalendar().week.astype(int)
    df['month']        = ts.dt.month

    # Impute distances with median
    for c in ['pickup_distance', 'dropoff_distance']:
        df[c].fillna(df[c].median(), inplace=True)
    df['total_distance'] = df['pickup_distance'] + df['dropoff_distance']
    df['distance_ratio'] = df['dropoff_distance'] / (df['pickup_distance'] + 0.001)
    df['log_total_dist'] = np.log1p(df['total_distance'])
    df['is_distance_missing'] = 0

    # Eater wait time
    df['eater_wait_min'] = (
        (df['restaurant_offered_timestamp_utc']
         - (df['eater_request_timestamp_local'] + pd.Timedelta(hours=6)))
        .dt.total_seconds() / 60
    ).clip(0, 60)
    df['eater_wait_min'].fillna(df['eater_wait_min'].median(), inplace=True)

    # Categorical codes
    for c in ['territory', 'courier_flow', 'geo_archetype', 'merchant_surface']:
        if df[c].dtype.name != 'category':
            df[c] = df[c].astype('category')
        df[f'{c}_code'] = df[c].cat.codes

    df['territory_median_atd'] = df.groupby('territory', observed=True)['ATD_capped'].transform('median')
    df['geo_archetype_median_atd'] = df.groupby('geo_archetype', observed=True)['ATD_capped'].transform('median')
    df['is_atd_outlier'] = 0
    print(f'Built features from raw: {df.shape}')

print(f'\nDate range: {df["restaurant_offered_timestamp_utc"].min()} → {df["restaurant_offered_timestamp_utc"].max()}')

---
## 2. Target Variable Analysis

In [ ]:
TARGET = 'ATD_capped'
y = df[TARGET]

print('Target variable (ATD_capped) stats:')
print(y.describe().round(2))
print(f'\nSkewness : {y.skew():.3f}')
print(f'Kurtosis : {y.kurt():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(y, bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(y.median(), color='red', linestyle='--', label=f'Median {y.median():.0f} min')
axes[0].legend()
axes[0].set_title('ATD Distribution (capped at p99)')
axes[0].set_xlabel('ATD (min)')

axes[1].hist(np.log1p(y), bins=80, color='teal', edgecolor='none')
axes[1].set_title('log(ATD+1) Distribution')
axes[1].set_xlabel('log(ATD + 1)')

# QQ-like: sorted values vs normal quantiles
from scipy import stats as scipy_stats
scipy_stats.probplot(y.sample(min(10_000, len(y)), random_state=SEED), plot=axes[2])
axes[2].set_title('Normal QQ Plot — ATD')

plt.suptitle('Target Variable Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Time-Based Train / Test Split

We use a **temporal split** to simulate how the model would be used in production:
- **Train**: all data except the last 14 days
- **Test**: the last 14 days

This prevents **data leakage** — a random split would allow future information to bleed into training.

In [ ]:
HOLDOUT_DAYS = 14

max_date   = df['restaurant_offered_timestamp_utc'].max()
split_date = max_date - pd.Timedelta(days=HOLDOUT_DAYS)

train_mask = df['restaurant_offered_timestamp_utc'] < split_date
test_mask  = df['restaurant_offered_timestamp_utc'] >= split_date

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

print(f'Split date         : {split_date.date()}')
print(f'Train rows         : {len(df_train):>10,}  ({len(df_train)/len(df)*100:.1f}%)')
print(f'Test  rows         : {len(df_test):>10,}  ({len(df_test)/len(df)*100:.1f}%)')
print(f'Train date range   : {df_train["restaurant_offered_timestamp_utc"].min().date()} → {df_train["restaurant_offered_timestamp_utc"].max().date()}')
print(f'Test  date range   : {df_test["restaurant_offered_timestamp_utc"].min().date()} → {df_test["restaurant_offered_timestamp_utc"].max().date()}')

# Check target distribution consistency
print(f'\nTrain ATD median   : {df_train[TARGET].median():.2f} min')
print(f'Test  ATD median   : {df_test[TARGET].median():.2f} min')

---
## 4. Feature Matrix Construction

In [ ]:
# Feature sets
NUMERIC_FEATURES = [
    'hour_utc', 'hour_local', 'day_of_week', 'is_weekend', 'is_peak_hour',
    'week_number', 'month',
    'pickup_distance', 'dropoff_distance', 'total_distance', 'distance_ratio',
    'log_total_dist',
    'is_long_trip', 'is_distance_missing',
    'eater_wait_min',
    'territory_median_atd', 'geo_archetype_median_atd',
]

CAT_FEATURES = [
    'territory', 'courier_flow', 'geo_archetype', 'merchant_surface',
]

# Filter to columns that exist in the dataframe
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in df.columns]
CAT_FEATURES     = [c for c in CAT_FEATURES if c in df.columns]

ALL_FEATURES = NUMERIC_FEATURES + CAT_FEATURES
print(f'Numeric features : {len(NUMERIC_FEATURES)}')
print(f'Categorical       : {len(CAT_FEATURES)}')
print(f'Total features    : {len(ALL_FEATURES)}')

# Ensure categoricals have proper type
for c in CAT_FEATURES:
    for dataset in [df_train, df_test]:
        if dataset[c].dtype.name != 'category':
            dataset[c] = dataset[c].astype('category')

X_train = df_train[ALL_FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[ALL_FEATURES]
y_test  = df_test[TARGET]

# Check for missing values in features
print(f'\nMissing values in X_train: {X_train.isnull().sum().sum()}')
print(f'Missing values in X_test : {X_test.isnull().sum().sum()}')

---
## 5. Baseline Models

We establish two baselines before training the ML model:
- **Global median**: predict the overall median ATD for every trip
- **Segment median**: predict the territory × courier_flow × time_block median ATD

In [ ]:
results = []

# Baseline 1: Global median
global_median = y_train.median()
pred_global   = np.full(len(y_test), global_median)
results.append(evaluate(y_test.values, pred_global, label='Baseline: Global Median'))

# Baseline 2: Territory × Courier Flow median
seg_median = (
    df_train.groupby(['territory', 'courier_flow'], observed=True)[TARGET]
    .median()
    .reset_index()
    .rename(columns={TARGET: 'seg_median'})
)
pred_seg = (
    df_test[['territory', 'courier_flow']]
    .merge(seg_median, on=['territory', 'courier_flow'], how='left')
    ['seg_median']
    .fillna(global_median)
    .values
)
results.append(evaluate(y_test.values, pred_seg, label='Baseline: Segment Median'))

# Baseline 3: Territory × Courier × Time Block median
if 'time_block' in df_train.columns:
    seg3_median = (
        df_train.groupby(['territory', 'courier_flow', 'time_block'], observed=True)[TARGET]
        .median()
        .reset_index()
        .rename(columns={TARGET: 'seg3_median'})
    )
    pred_seg3 = (
        df_test[['territory', 'courier_flow', 'time_block']]
        .merge(seg3_median, on=['territory', 'courier_flow', 'time_block'], how='left')
        ['seg3_median']
        .fillna(pred_seg)
        .values
    )
    results.append(evaluate(y_test.values, pred_seg3, label='Baseline: Seg×Time Median'))

---
## 6. LightGBM Model

**Why LightGBM?**
- Handles 1M+ rows efficiently (leaf-wise tree growth)
- Native categorical support (no one-hot encoding needed)
- Built-in feature importance and SHAP compatibility
- Robust to outliers with `huber` or `quantile` objectives
- Achieves best accuracy vs training time trade-off for tabular delivery data

In [ ]:
lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=CAT_FEATURES,
    free_raw_data=False,
)
lgb_valid = lgb.Dataset(
    X_test, label=y_test,
    categorical_feature=CAT_FEATURES,
    reference=lgb_train,
    free_raw_data=False,
)

params = {
    'objective':        'regression_l1',   # MAE-optimized (robust to remaining outliers)
    'metric':           ['mae', 'rmse'],
    'learning_rate':    0.05,
    'num_leaves':       127,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'verbose':          -1,
    'seed':             SEED,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100),
]

print('Training LightGBM model (MAE objective)...')
lgb_model = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    callbacks=callbacks,
)

print(f'\nBest iteration: {lgb_model.best_iteration}')

In [ ]:
pred_lgbm = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
pred_lgbm = np.clip(pred_lgbm, a_min=0, a_max=None)  # predictions must be non-negative

result_lgbm = evaluate(y_test.values, pred_lgbm, label='LightGBM')
results.append(result_lgbm)

---
## 7. Evaluation & Benchmarking

In [ ]:
results_df = pd.DataFrame(results).set_index('label').round(3)
print('\n── Model Comparison ──')
display(results_df.sort_values('MAE'))

# Improvement over best baseline
best_baseline_mae = results_df.loc[
    [r for r in results_df.index if 'Baseline' in r], 'MAE'
].min()
lgbm_mae = results_df.loc['LightGBM', 'MAE']
improvement = (best_baseline_mae - lgbm_mae) / best_baseline_mae * 100
print(f'\nLightGBM MAE improvement over best baseline: {improvement:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# MAE comparison bar
results_df['MAE'].sort_values(ascending=True).plot(
    kind='barh', ax=axes[0],
    color=['seagreen' if 'LightGBM' in i else 'steelblue' for i in results_df.sort_values('MAE').index],
    edgecolor='none',
)
axes[0].set_title('MAE Comparison')
axes[0].set_xlabel('MAE (min)')

# Predicted vs actual (sample)
sample_idx = np.random.choice(len(y_test), size=min(5000, len(y_test)), replace=False)
y_sample    = y_test.values[sample_idx]
pred_sample = pred_lgbm[sample_idx]
axes[1].scatter(y_sample, pred_sample, alpha=0.05, s=5, color='steelblue')
max_val = max(y_sample.max(), pred_sample.max())
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect fit')
axes[1].set_xlabel('Actual ATD (min)')
axes[1].set_ylabel('Predicted ATD (min)')
axes[1].set_title('LightGBM: Predicted vs Actual')
axes[1].legend()

# Residual distribution
residuals = y_test.values - pred_lgbm
axes[2].hist(residuals, bins=80, color='coral', edgecolor='none')
axes[2].axvline(0, color='black', linewidth=1)
axes[2].axvline(np.median(residuals), color='red', linestyle='--',
                label=f'Median residual: {np.median(residuals):.2f} min')
axes[2].set_title('LightGBM Residual Distribution')
axes[2].set_xlabel('Actual − Predicted (min)')
axes[2].legend()

plt.suptitle('Model Evaluation', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate by segment
df_eval = df_test.copy()
df_eval['predicted'] = pred_lgbm
df_eval['residual']  = df_eval[TARGET] - df_eval['predicted']
df_eval['abs_error'] = df_eval['residual'].abs()

for seg_col in ['territory', 'courier_flow', 'time_block' if 'time_block' in df.columns else 'is_peak_hour']:
    if seg_col not in df_eval.columns:
        continue
    seg_eval = (
        df_eval.groupby(seg_col, observed=True)
        .agg(
            n=('abs_error', 'count'),
            mae=('abs_error', 'mean'),
            median_residual=('residual', 'median'),
        )
        .round(2)
        .sort_values('mae', ascending=False)
    )
    print(f'\nMAE by {seg_col}:')
    display(seg_eval)

---
## 8. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': lgb_model.feature_name(),
    'gain':    lgb_model.feature_importance(importance_type='gain'),
    'split':   lgb_model.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

importance_df['gain_pct']  = importance_df['gain']  / importance_df['gain'].sum()  * 100
importance_df['split_pct'] = importance_df['split'] / importance_df['split'].sum() * 100

print('Top 20 features by gain:')
display(importance_df.head(20).round(2))

In [ ]:
top_n = 20
top_features = importance_df.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top_features.set_index('feature')['gain_pct'].sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title(f'Top {top_n} Features — Gain (%)', fontweight='bold')
axes[0].set_xlabel('% of Total Gain')

top_features.set_index('feature')['split_pct'].sort_values().plot(
    kind='barh', ax=axes[1], color='coral', edgecolor='none')
axes[1].set_title(f'Top {top_n} Features — Split Count (%)', fontweight='bold')
axes[1].set_xlabel('% of Total Splits')

plt.suptitle('LightGBM Feature Importance', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Residual & Error Analysis

Understanding where the model fails helps inform operational priorities.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Residual vs predicted (heteroscedasticity check)
axes[0, 0].scatter(pred_lgbm, residuals, alpha=0.03, s=4, color='steelblue')
axes[0, 0].axhline(0, color='red', linewidth=1)
axes[0, 0].set_xlabel('Predicted ATD (min)')
axes[0, 0].set_ylabel('Residual (min)')
axes[0, 0].set_title('Residuals vs Predicted')

# 2. Absolute error by ATD bucket
atd_buckets = pd.cut(y_test, bins=[0, 15, 30, 45, 60, 90, 120, np.inf],
                     labels=['0-15', '15-30', '30-45', '45-60', '60-90', '90-120', '120+'])
mae_by_bucket = df_eval.groupby(atd_buckets, observed=True)['abs_error'].mean()
mae_by_bucket.plot(kind='bar', ax=axes[0, 1], color='coral', edgecolor='none')
axes[0, 1].set_title('MAE by ATD Bucket')
axes[0, 1].set_xlabel('Actual ATD Range (min)')
axes[0, 1].set_ylabel('MAE (min)')
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. MAE by hour
if 'hour_local' in df_eval.columns:
    mae_hour = df_eval.groupby('hour_local')['abs_error'].mean()
    mae_hour.plot(kind='line', ax=axes[1, 0], color='teal', marker='o', linewidth=2)
    axes[1, 0].set_title('MAE by Hour (Local)')
    axes[1, 0].set_xlabel('Local Hour')
    axes[1, 0].set_ylabel('MAE (min)')
    axes[1, 0].set_xticks(range(0, 24))

# 4. Error distribution tail analysis
high_error = (df_eval['abs_error'] > 20)
axes[1, 1].hist(df_eval.loc[high_error, 'abs_error'], bins=50, color='tomato', edgecolor='none')
axes[1, 1].set_title(f'High Error Distribution (> 20 min) — {high_error.mean()*100:.1f}% of trips')
axes[1, 1].set_xlabel('Absolute Error (min)')

plt.suptitle('Residual & Error Analysis', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\nErrors > 20 min: {high_error.sum():,} trips ({high_error.mean()*100:.1f}%)')
print('Segment breakdown of high-error trips:')
display(
    df_eval[high_error].groupby('courier_flow', observed=True).size()
    .sort_values(ascending=False)
    .rename('high_error_count')
    .to_frame()
    .assign(pct=lambda x: (x['high_error_count'] / high_error.sum() * 100).round(1))
)

In [ ]:
# Show concrete prediction examples from test set
show_cols = ['territory', 'courier_flow', 'hour_local', 'total_distance', TARGET, 'predicted', 'abs_error']
show_cols = [c for c in show_cols if c in df_eval.columns]

print('10 random test predictions:')
display(df_eval[show_cols].sample(10, random_state=SEED).round(2))

print('\n10 worst predictions (highest absolute error):')
display(df_eval[show_cols].nlargest(10, 'abs_error').round(2))

---
## 10. Model Persistence & Summary

In [ ]:
# Save model
model_path = MODEL_DIR / 'lgbm_atd_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(lgb_model, f)
print(f'Model saved → {model_path}')

# Save feature list and metadata
model_meta = {
    'feature_cols': ALL_FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'cat_features': CAT_FEATURES,
    'target': TARGET,
    'split_date': str(split_date.date()),
    'best_iteration': lgb_model.best_iteration,
    'test_mae':  result_lgbm['MAE'],
    'test_rmse': result_lgbm['RMSE'],
    'test_mape': result_lgbm['MAPE'],
    'test_r2':   result_lgbm['R2'],
}

import json
meta_path = MODEL_DIR / 'model_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(model_meta, f, indent=2)
print(f'Metadata saved → {meta_path}')

In [ ]:
print('═' * 60)
print('  PREDICTIVE MODELING SUMMARY')
print('═' * 60)
print(f'  Dataset               : {len(df):,} trips ({len(df_train):,} train, {len(df_test):,} test)')
print(f'  Features              : {len(ALL_FEATURES)} ({len(NUMERIC_FEATURES)} numeric + {len(CAT_FEATURES)} categorical)')
print(f'  Model                 : LightGBM (MAE objective)')
print(f'  Best iteration        : {lgb_model.best_iteration}')
print()
print(f'  ─── Test Set Performance ───')
for r in results:
    print(f'  {r["label"]:<35}  MAE={r["MAE"]:.2f}  RMSE={r["RMSE"]:.2f}  MAPE={r["MAPE"]:.1f}%  R²={r["R2"]:.4f}')
print()
print(f'  LightGBM improvement vs best baseline:')
print(f'    MAE  reduction: {improvement:.1f}%')
print()
print(f'  Top 5 features (by gain):')
for _, row in importance_df.head(5).iterrows():
    print(f'    {row["feature"]:<30}  {row["gain_pct"]:.1f}% of gain')
print('═' * 60)